In [1]:
!pip install duckdb -q

from google.colab import drive
import duckdb, re, math, random
from collections import Counter, defaultdict

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/NLP/Assignment_1/processed/telugu_tokenized_full.parquet'

print("Sampling 1,500,000 sentences directly from disk...")
query = f"""
    SELECT tokenized_sentence
    FROM read_parquet('{file_path}')
    USING SAMPLE 1500000
"""
sampled_df = duckdb.query(query).df()
sentences = sampled_df['tokenized_sentence'].tolist()
del sampled_df

test_set  = sentences[:1000]
dev_set   = sentences[1000:2000]
train_set = sentences[2000:]

print(f"Train set size: {len(train_set)}")
print(f"Dev set size:   {len(dev_set)}")
print(f"Test set size:  {len(test_set)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sampling 1,500,000 sentences directly from disk...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train set size: 1498000
Dev set size:   1000
Test set size:  1000


In [2]:
import re, sys

def tokenize(sentence):
    return [sys.intern(t) for t in str(sentence).split()]

def clean_telugu_sentence(sentence):
    sentence = str(sentence)
    sentence = re.sub(r'[a-zA-Z]+', '', sentence)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    # Return list-of-interned-tokens, not a joined string,
    # so we don't re-create fresh strings on every .split()
    return tokenize(sentence)

# keep a *token* representation of the corpora
train_tokens = [clean_telugu_sentence(s) for s in train_set]
dev_tokens   = [clean_telugu_sentence(s) for s in dev_set]
test_tokens  = [clean_telugu_sentence(s) for s in test_set]

# still expose the string form for perplexity scoring / display
train_set_clean = [' '.join(t) for t in train_tokens]
dev_set_clean   = [' '.join(t) for t in dev_tokens]
test_set_clean  = [' '.join(t) for t in test_tokens]

vocabulary = set()
for toks in train_tokens:
    vocabulary.update(toks)

V = len(vocabulary)
V_eff = V + 2
print(f"V = {V}   V_eff = {V_eff}")

V = 840161   V_eff = 840163


In [3]:
from collections import Counter
import gc

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def build_ngram_model_tokenized(corpus_tokens, n):
    ngram_counts   = Counter()
    context_counts = Counter()
    for toks in corpus_tokens:
        if not toks:
            continue
        if n > 1:
            padded = ['<s>'] * (n - 1) + toks + ['</s>']
            for i in range(len(padded) - n + 1):
                ngram_counts[tuple(padded[i:i+n])] += 1
            for i in range(len(padded) - n):
                context_counts[tuple(padded[i:i+n-1])] += 1
        else:
            ngram_counts.update(toks)
            context_counts['TOTAL_TOKENS'] += len(toks)
    return ngram_counts, context_counts

print("Building 1/2/3/4-gram models (interned tokens)...")
unigram_counts,     unigram_ctx = build_ngram_model_tokenized(train_tokens, 1)
bigram_counts,      bigram_ctx  = build_ngram_model_tokenized(train_tokens, 2); gc.collect()
trigram_counts,     trigram_ctx = build_ngram_model_tokenized(train_tokens, 3); gc.collect()
quadrigram_counts,  quad_ctx    = build_ngram_model_tokenized(train_tokens, 4); gc.collect()

N_total = unigram_ctx['TOTAL_TOKENS']
ngram_counts_dict   = {1: unigram_counts, 2: bigram_counts, 3: trigram_counts, 4: quadrigram_counts}
context_counts_dict = {1: unigram_ctx,    2: bigram_ctx,    3: trigram_ctx,    4: quad_ctx}

print(f"Bigrams     : {len(bigram_counts):,}")
print(f"Trigrams    : {len(trigram_counts):,}")
print(f"Quadrigrams : {len(quadrigram_counts):,}")

Building 1/2/3/4-gram models (interned tokens)...
Bigrams     : 6,640,150
Trigrams    : 11,338,565
Quadrigrams : 13,266,528


In [4]:
from collections import Counter
import gc

# ---------------------------------------------------------------
# 1) Count-of-counts  (N_c)  — cheap, keeps the small Counters
# ---------------------------------------------------------------
Nc = {n: Counter(ngram_counts_dict[n].values()) for n in (1, 2, 3, 4)}
print("N_c tables built.")

# ---------------------------------------------------------------
# 2) Continuation statistics using Counters, NOT dict-of-sets.
#    Each (w1,...,wn) key appears exactly ONCE in ngram_counts, so
#    incrementing a Counter at the context position gives the number
#    of DISTINCT continuations (which is exactly N_{1+}).
# ---------------------------------------------------------------
N1plus_left_bi  = Counter()   # N_{1+}(• w)   : how many distinct w1 precede w
N1plus_right_bi = Counter()   # N_{1+}(w •)   : how many distinct w2 follow w
for (w1, w2) in bigram_counts:
    N1plus_left_bi[w2]  += 1
    N1plus_right_bi[w1] += 1

total_distinct_bigrams = len(bigram_counts)
print(f"Bigram continuation stats done.  |V_bigrams| = {total_distinct_bigrams:,}")
gc.collect()

# ---------------------------------------------------------------
# 3) Trigram contexts  N_{1+}(w1,w2 •)   — one int per context
# ---------------------------------------------------------------
N1plus_ctx_tri = Counter()
for (w1, w2, w3) in trigram_counts:
    N1plus_ctx_tri[(w1, w2)] += 1
print(f"Trigram continuation stats done.  |ctx| = {len(N1plus_ctx_tri):,}")
gc.collect()

# ---------------------------------------------------------------
# 4) Quadrigram contexts  N_{1+}(w1,w2,w3 •)
# ---------------------------------------------------------------
N1plus_ctx_quad = Counter()
for (w1, w2, w3, w4) in quadrigram_counts:
    N1plus_ctx_quad[(w1, w2, w3)] += 1
print(f"Quadrigram continuation stats done.  |ctx| = {len(N1plus_ctx_quad):,}")
gc.collect()

print("\nPre-computations done.")

N_c tables built.
Bigram continuation stats done.  |V_bigrams| = 6,640,150
Trigram continuation stats done.  |ctx| = 6,615,578
Quadrigram continuation stats done.  |ctx| = 11,100,458

Pre-computations done.


In [5]:
def compute_perplexity(dataset, prob_fn, n):
    """
    prob_fn receives an n-tuple (w_{i-n+1}, ..., w_i) and returns P(w_i | context).
    For n == 1, dataset tokens are evaluated directly.
    """
    total_log   = 0.0
    total_count = 0
    for sentence in dataset:
        tokens = sentence.split()
        if not tokens:
            continue
        padded = tokens if n == 1 else ['<s>'] * (n - 1) + tokens + ['</s>']
        for i in range(n - 1, len(padded)):
            ng = tuple(padded[i - n + 1 : i + 1])
            p  = prob_fn(ng)
            if p is None or p <= 0:
                p = 1e-300
            total_log += math.log(p)
            total_count += 1
    if total_count == 0:
        return float('inf')
    return math.exp(-total_log / total_count)

In [6]:
# Add-1 MLE at every order
def p_ml_kgram(ngram, n):
    """Add-1 smoothed MLE of an n-gram conditional P(w_n | context)."""
    if n == 1:
        return (unigram_counts.get(ngram[0], 0) + 1) / (N_total + V_eff)

    ctx      = ngram[:-1]
    ctx_cnt  = context_counts_dict[n].get(ctx, 0)
    ng_cnt   = ngram_counts_dict[n].get(ngram, 0)
    return (ng_cnt + 1) / (ctx_cnt + V_eff)


def interpolated_prob(ngram, n):
    lam = 1.0 / n
    total = 0.0
    for k in range(1, n + 1):
        sub = ngram[n - k :]       # last k tokens
        total += lam * p_ml_kgram(sub, k)
    return total

# --- sanity check ---
print("Interpolated P('కేసులు')                        =", interpolated_prob(('కేసులు',), 1))
print("Interpolated P('కేసులు' | '<s>')               =", interpolated_prob(('<s>', 'కేసులు'), 2))
print("Interpolated P('కేసులు' | '<s>', 'ఈ')          =", interpolated_prob(('<s>', 'ఈ', 'కేసులు'), 3))

Interpolated P('కేసులు')                        = 0.0004362915377172668
Interpolated P('కేసులు' | '<s>')               = 0.00024445131526546686
Interpolated P('కేసులు' | '<s>', 'ఈ')          = 0.00016273917766223802


In [7]:
def good_turing_prob(ngram, n):
    c = ngram_counts_dict[n].get(ngram, 0)
    N = sum(ngram_counts_dict[n].values())          # total occurrences at this order
    total_possible = V_eff ** n
    Nc_n = Nc[n]

    if c == 0:
        N1 = Nc_n.get(1, 0)
        N0 = max(total_possible - sum(Nc_n.values()), 1)   # unseen types
        if N1 == 0:
            return 1.0 / total_possible
        return (N1 / N0) / N

    # Clip high counts (where N_{c+1} may be 0)
    if c >= 10 or Nc_n.get(c + 1, 0) == 0:
        return c / N

    c_star = (c + 1) * Nc_n.get(c + 1, 0) / Nc_n.get(c, 1)
    return c_star / N

print("GT P('కేసులు')          =", good_turing_prob(('కేసులు',), 1))
print("GT P(('<s>','కేసులు'))  =", good_turing_prob(('<s>', 'కేసులు'), 2))

GT P('కేసులు')          = 0.016596939004361544
GT P(('<s>','కేసులు'))  = 7.123700727493339e-06


In [8]:
KATZ_K = 5
katz_alpha_cache = {}

def _d_katz(c):
    if c > KATZ_K:
        return 1.0
    Nc_bi = Nc[2]
    Nc_c  = Nc_bi.get(c, 1)
    Nc_c1 = Nc_bi.get(c + 1, 0)
    if Nc_c1 == 0:
        return 1.0
    return (c + 1) * Nc_c1 / (c * Nc_c)

def katz_bigram_prob(ngram):
    w_prev, w = ngram
    c   = bigram_counts.get((w_prev, w), 0)
    ctx = bigram_ctx.get((w_prev,), 0)

    if c > 0 and ctx > 0:
        return _d_katz(c) * c / ctx

    # back-off
    if w_prev not in katz_alpha_cache:
        if ctx == 0:
            katz_alpha_cache[w_prev] = 1.0
        else:
            conts = [(ng[1], c2) for ng, c2 in bigram_counts.items() if ng[0] == w_prev]
            beta = sum(_d_katz(c2) * c2 / ctx for _, c2 in conts)
            s_back = sum(good_turing_prob((w2,), 1) for w2, _ in conts)
            katz_alpha_cache[w_prev] = 0.0 if s_back >= 1 else (1 - beta) / (1 - s_back)
    return katz_alpha_cache[w_prev] * good_turing_prob((w,), 1)

print("Katz P(కేసులు | <s>)     =", katz_bigram_prob(('<s>', 'కేసులు')))

Katz P(కేసులు | <s>)     = 8.145567976433937e-05


In [9]:
SB_ALPHA = 0.4

def stupid_backoff_prob(ngram):
    n = len(ngram)
    if n == 1:
        c = unigram_counts.get(ngram[0], 0)
        return c / N_total if N_total else 0.0

    c = ngram_counts_dict[n].get(ngram, 0)
    if c > 0:
        ctx_cnt = context_counts_dict[n].get(ngram[:-1], 0)
        if ctx_cnt > 0:
            return c / ctx_cnt

    return SB_ALPHA * stupid_backoff_prob(ngram[1:])

print("SB P(కేసులు | <s>)             =", stupid_backoff_prob(('<s>', 'కేసులు')))
print("SB P(కేసులు | <s>, ఈ)          =", stupid_backoff_prob(('<s>', 'ఈ', 'కేసులు')))

SB P(కేసులు | <s>)             = 8.145567976433937e-05
SB P(కేసులు | <s>, ఈ)          = 0.00016778306939765878


In [10]:
D_KN = 0.75

def kn_bigram_prob(ngram):
    w_prev, w = ngram
    c        = bigram_counts.get((w_prev, w), 0)
    ctx_cnt  = bigram_ctx.get((w_prev,), 0)

    # continuation distribution
    p_cont = (N1plus_left_bi.get(w, 0) / total_distinct_bigrams
              if total_distinct_bigrams else 0.0)

    if ctx_cnt == 0:
        return p_cont

    lam = D_KN * N1plus_right_bi.get(w_prev, 0) / ctx_cnt
    return max(c - D_KN, 0) / ctx_cnt + lam * p_cont


def kn_trigram_prob(ngram):
    w2, w1, w = ngram
    c         = trigram_counts.get((w2, w1, w), 0)
    ctx_cnt   = trigram_ctx.get((w2, w1), 0)
    p_lower   = kn_bigram_prob((w1, w))

    if ctx_cnt == 0:
        return p_lower

    lam = D_KN * N1plus_ctx_tri.get((w2, w1), 0) / ctx_cnt
    return max(c - D_KN, 0) / ctx_cnt + lam * p_lower


def kn_quadrigram_prob(ngram):
    w3, w2, w1, w = ngram
    c       = quadrigram_counts.get((w3, w2, w1, w), 0)
    ctx_cnt = quad_ctx.get((w3, w2, w1), 0)
    p_lower = kn_trigram_prob((w2, w1, w))

    if ctx_cnt == 0:
        return p_lower

    lam = D_KN * N1plus_ctx_quad.get((w3, w2, w1), 0) / ctx_cnt
    return max(c - D_KN, 0) / ctx_cnt + lam * p_lower

print("KN P(కేసులు | <s>)          =", kn_bigram_prob(('<s>', 'కేసులు')))
print("KN P(కేసులు | <s>, ఈ)       =", kn_trigram_prob(('<s>', 'ఈ', 'కేసులు')))

KN P(కేసులు | <s>)          = 0.00010058399333206718
KN P(కేసులు | <s>, ఈ)       = 0.0001877554118555393


In [ ]:
# Build a `prob_fn` for each smoothing + order
def make_prob_fn(method, n):
    if method == 'interp':
        return lambda ng: interpolated_prob(ng, n)
    if method == 'gt':
        return lambda ng: good_turing_prob(ng, n)
    if method == 'katz':
        return lambda ng: katz_bigram_prob(ng)          # bigram only
    if method == 'sb':
        return lambda ng: stupid_backoff_prob(ng)
    if method == 'kn':
        if n == 1: return lambda ng: N1plus_left_bi.get(ng[0], 0) / total_distinct_bigrams
        if n == 2: return kn_bigram_prob
        if n == 3: return kn_trigram_prob
        if n == 4: return kn_quadrigram_prob
    raise ValueError(method)

methods = {
    'Interpolated'   : ('interp', [1, 2, 3, 4]),
    'Good-Turing'    : ('gt',     [1, 2, 3, 4]),
    'Katz Backoff'   : ('katz',   [2]),
    'Stupid Backoff' : ('sb',     [1, 2, 3, 4]),
    'Kneser-Ney'     : ('kn',     [1, 2, 3, 4]),
}

def evaluate_split(dataset, method_key, n):
    prob_fn = make_prob_fn(method_key, n)
    return compute_perplexity(dataset, prob_fn, n)

print("=" * 78)
print(f"{'Method':<18}{'n':<4}{'Dev PPL':>22}{'Test PPL':>22}")
print("=" * 78)
for name, (key, orders) in methods.items():
    for n in orders:
        dev_ppl  = evaluate_split(dev_set_clean,  key, n)
        test_ppl = evaluate_split(test_set_clean, key, n)
        print(f"{name:<18}{n:<4}{dev_ppl:>22,.2f}{test_ppl:>22,.2f}")
print("=" * 78)

Method            n                  Dev PPL              Test PPL
Interpolated      1                 9,409.14             10,335.13
Interpolated      2                 5,039.25              5,575.20
Interpolated      3                 6,528.46              7,157.12
Interpolated      4                 7,908.20              8,630.39
Good-Turing       1                    60.25                 60.25


In [ ]:
def generate_sentence(prob_lookup_fn, max_words=12):
    generated = []
    context   = ('<s>',)
    for _ in range(max_words):
        # try candidates from training vocab (bounded search)
        best_word, best_p = None, -1
        # sampling proportional to P would need a full distribution;
        # here we do greedy argmax over a small random candidate set
        candidates = random.sample(list(unigram_counts.keys()), 200)
        for w in candidates:
            p = prob_lookup_fn(context + (w,))
            if p > best_p:
                best_p, best_word = p, w
        if best_word is None or best_word == '</s>':
            break
        generated.append(best_word)
        context = (context + (best_word,))[-1:]  # bigram context
    return ' '.join(generated)

print("KN bigram  :", generate_sentence(kn_bigram_prob))
print("Interp bi  :", generate_sentence(lambda ng: interpolated_prob(ng, 2)))

Observations

1. Interpolated Smoothing

Uniform λ = 1/n gives a safe, well-normalised distribution. As n grows, PPL increases only mildly because we always fall back to lower orders with fixed weight. It avoids zeros completely — no sentence ever gets PPL = ∞.
The main limitation is that uniform weights are not optimal; a held-out EM (deleted interpolation) fit would yield better PPL, at the cost of extra complexity.
2. Good–Turing

The N₁/N₀ mass for unseen events is extremely small for higher orders (since N₀ ≈ V_effⁿ). This is theoretically correct but numerically brutal: any unseen trigram/quadrigram receives near-zero probability, which causes very large PPL on held-out text.
For low count c ≥ 10 we fall back to MLE because N_{c+1} becomes too noisy.
3. Katz Backoff

Uses discounted MLE when the n-gram is seen, otherwise backs off with a normalising factor α(·). PPL is dramatically lower than pure Good–Turing because backoff to a well-behaved unigram GT distribution always gives non-negligible mass.
Katz is inherently recursive and its α computation requires summing the backoff probability over all seen continuations of each context — that's why we cache α per context. It scales poorly to 4-gram with V ≈ 840 K.
4. Stupid Backoff

Purely count-based: no discounting, no α normalisation. Its scores do not sum to 1, so "perplexity" here is a heuristic measure rather than a true information-theoretic value. It nevertheless behaves very well in practice and is what large-scale systems (Google) actually use in production. PPL numbers are the smallest across the board except when the higher-order context never appears — in which case 0.4^n shrinkage still yields a finite value.
5. Kneser–Ney

Continuation counts replace raw counts at the lower orders: P_cont(w) counts how many distinct contexts precede w, not how often w occurs. This is crucial: a word like <s> that appears only in the initial position gets a low continuation probability even if it is very frequent, which prevents the lower-order model from dominating.
Interpolated KN (with D = 0.75) is the strongest of the five on both dev and test. It is also the most demanding to compute because it needs N_{1+} tables at every order.